# Week 2 · Activity 1: Stock Data Pipeline & Analysis
### eMathrix Education — Data Science & Data Analytics

In this activity you build an automated **data pipeline** with **n8n** that pulls daily stock data from the **Polygon.io** API, scores it, and appends it to a **Google Sheet** — then you analyze that collected data here in Colab.

**The n8n pipeline (Steps 1-5) is documented in the Activity 1 n8n guide.** This notebook is the **analysis** half: it reads the Google Sheet your pipeline fills and turns it into charts and insights.

Tickers used in the demo: AAPL, META, MSFT, GOOGL, TSLA.

## Step 1: Point this at your own Google Sheet

Your n8n pipeline writes rows into a Google Sheet. Make that sheet **viewable by anyone with the link**, copy its **Sheet ID** from the URL, and paste it where you see `PASTE_YOUR_SHEET_ID_HERE` below.

If the sheet can't be reached, the code falls back to **sample data** so the analysis still runs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


def load_google_sheet_data(sheet_url=None, sheet_id=None):
    """
    Load data from Google Sheets with proper error handling
    """
    print("Loading data from Google Sheets...")

    # Method 1: If full URL is provided
    if sheet_url:
        if '/d/' in sheet_url:
            sheet_id = sheet_url.split('/d/')[1].split('/')[0]
            print(f"Extracted Sheet ID: {sheet_id}")

    # Use your own sheet ID here
    if not sheet_id:
        sheet_id = 'PASTE_YOUR_SHEET_ID_HERE'

    methods = [
        {'name': 'Export as CSV',
         'url': f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv'},
        {'name': 'GViz Query (CSV)',
         'url': f'https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv'},
        {'name': 'Published CSV',
         'url': f'https://docs.google.com/spreadsheets/d/e/{sheet_id}/pub?output=csv'},
    ]

    df = None
    for method in methods:
        try:
            print(f"Trying method: {method['name']}...")
            df = pd.read_csv(method['url'])
            print(f"Success! Loaded {len(df)} rows using {method['name']}")
            break
        except Exception as e:
            print(f"{method['name']} failed: {str(e)[:50]}...")
            continue

    if df is None:
        print("\nCould not load from Google Sheets. Using sample data instead...")
        df = create_sample_stock_data()

    return df


def create_sample_stock_data():
    """
    Create sample stock data for testing when sheet is not accessible
    """
    print("Creating sample stock data for analysis...")

    dates = pd.date_range(start='2025-07-26', periods=100, freq='h')

    np.random.seed(42)
    base_price = 213.88
    prices = [base_price]

    for i in range(1, len(dates)):
        change = np.random.normal(0, 0.5)
        new_price = prices[-1] * (1 + change/100)
        prices.append(new_price)

    returns = [0] + [(prices[i] - prices[i-1])/prices[i-1] * 100 for i in range(1, len(prices))]

    signals = []
    risk_levels = []
    for ret in returns:
        if ret > 0.5:
            signals.append('BUY'); risk_levels.append('HIGH')
        elif ret < -0.5:
            signals.append('SELL'); risk_levels.append('HIGH')
        else:
            signals.append('HOLD'); risk_levels.append('LOW')

    df = pd.DataFrame({
        'DATE': dates,
        'SYMBOL': 'AAPL',
        'CLOSE PRICE': prices,
        'DAILY RETURN %': returns,
        'SIGNAL': signals,
        'RISK LEVEL': risk_levels
    })
    return df


def analyze_stock_data(df):
    """
    Comprehensive analysis and visualization of stock data
    """
    print("\nSTOCK DATA ANALYSIS")
    print("="*60)

    if df is None or len(df) == 0:
        print("No data to analyze")
        return None

    print(f"Analyzing {len(df)} data points")
    print(f"Columns found: {list(df.columns)}")

    df.columns = df.columns.str.strip()

    if 'DATE' in df.columns:
        df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')
    if 'CLOSE PRICE' in df.columns:
        df['CLOSE PRICE'] = pd.to_numeric(df['CLOSE PRICE'], errors='coerce')
    if 'DAILY RETURN %' in df.columns:
        if df['DAILY RETURN %'].dtype == 'object':
            df['DAILY RETURN %'] = df['DAILY RETURN %'].str.rstrip('%').astype('float', errors='ignore')
        df['DAILY RETURN %'] = pd.to_numeric(df['DAILY RETURN %'], errors='coerce')

    if 'DATE' in df.columns:
        df = df.sort_values('DATE')

    create_comprehensive_visualizations(df)
    return df


def create_comprehensive_visualizations(df):
    """
    Create multiple visualization types for stock data
    """
    print("\nCreating Interactive Price Chart...")

    subplot_titles = []
    specs = []
    if 'CLOSE PRICE' in df.columns:
        subplot_titles.append('Stock Price Over Time'); specs.append([{'type': 'scatter'}])
    if 'DAILY RETURN %' in df.columns:
        subplot_titles.append('Daily Returns %'); specs.append([{'type': 'bar'}])
    if 'SIGNAL' in df.columns:
        subplot_titles.append('Signal Distribution'); specs.append([{'type': 'bar'}])

    if len(subplot_titles) > 0:
        fig = make_subplots(rows=len(subplot_titles), cols=1,
                            subplot_titles=subplot_titles, vertical_spacing=0.1, specs=specs)
        current_row = 1

        if 'CLOSE PRICE' in df.columns and 'DATE' in df.columns:
            fig.add_trace(go.Scatter(x=df['DATE'], y=df['CLOSE PRICE'], mode='lines',
                                     name='Close Price', line=dict(color='blue', width=2)),
                          row=current_row, col=1)
            if 'SIGNAL' in df.columns:
                for signal in df['SIGNAL'].unique():
                    if pd.notna(signal):
                        signal_df = df[df['SIGNAL'] == signal]
                        color = 'green' if signal == 'BUY' else 'red' if signal == 'SELL' else 'gray'
                        fig.add_trace(go.Scatter(x=signal_df['DATE'], y=signal_df['CLOSE PRICE'],
                                                 mode='markers', name=f'{signal} Signal',
                                                 marker=dict(color=color, size=8, symbol='circle')),
                                      row=current_row, col=1)
            current_row += 1

        if 'DAILY RETURN %' in df.columns and 'DATE' in df.columns:
            returns = df['DAILY RETURN %'].fillna(0)
            colors = ['green' if x > 0 else 'red' for x in returns]
            fig.add_trace(go.Bar(x=df['DATE'], y=returns, name='Daily Return %', marker_color=colors),
                          row=current_row, col=1)
            current_row += 1

        if 'SIGNAL' in df.columns:
            signal_counts = df['SIGNAL'].value_counts()
            fig.add_trace(go.Bar(x=signal_counts.index, y=signal_counts.values, name='Signal Count',
                                 marker_color=['green' if x == 'BUY' else 'red' if x == 'SELL' else 'gray'
                                               for x in signal_counts.index]),
                          row=current_row, col=1)

        symbol = df['SYMBOL'].iloc[0] if 'SYMBOL' in df.columns and len(df) > 0 else 'Stock'
        fig.update_layout(height=300 * len(subplot_titles), showlegend=True,
                          title_text=f"Stock Analysis Dashboard - {symbol}", hovermode='x unified')
        fig.show()

    print("\nCreating Static Analysis Charts...")
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    symbol = df['SYMBOL'].iloc[0] if 'SYMBOL' in df.columns and len(df) > 0 else 'Stock'
    fig.suptitle(f'Stock Analysis - {symbol}', fontsize=16, fontweight='bold')

    ax1 = axes[0, 0]
    if 'DATE' in df.columns and 'CLOSE PRICE' in df.columns:
        ax1.plot(df['DATE'], df['CLOSE PRICE'], label='Close Price', color='blue', linewidth=2)
        if len(df) > 20:
            df['MA20'] = df['CLOSE PRICE'].rolling(window=20).mean()
            ax1.plot(df['DATE'], df['MA20'], label='20-day MA', color='orange', linestyle='--')
        ax1.set_xlabel('Date'); ax1.set_ylabel('Price ($)'); ax1.set_title('Price Trend')
        ax1.legend(); ax1.grid(True, alpha=0.3); ax1.tick_params(axis='x', rotation=45)

    ax2 = axes[0, 1]
    if 'DAILY RETURN %' in df.columns:
        returns_clean = df['DAILY RETURN %'].dropna()
        if len(returns_clean) > 0:
            ax2.hist(returns_clean, bins=min(30, len(returns_clean)),
                     color='skyblue', edgecolor='black', alpha=0.7)
            mean_return = returns_clean.mean()
            ax2.axvline(mean_return, color='red', linestyle='--', label=f'Mean: {mean_return:.2f}%')
            ax2.set_xlabel('Daily Return %'); ax2.set_ylabel('Frequency')
            ax2.set_title('Returns Distribution'); ax2.legend()

    ax3 = axes[0, 2]
    if 'RISK LEVEL' in df.columns:
        risk_counts = df['RISK LEVEL'].value_counts()
        colors_map = {'LOW': 'green', 'MEDIUM': 'yellow', 'HIGH': 'red'}
        colors = [colors_map.get(x, 'gray') for x in risk_counts.index]
        ax3.pie(risk_counts.values, labels=risk_counts.index, colors=colors,
                autopct='%1.1f%%', startangle=90)
        ax3.set_title('Risk Level Distribution')
    else:
        ax3.text(0.5, 0.5, 'Risk Level\nNot Available', ha='center', va='center', fontsize=12)
        ax3.set_xlim(0, 1); ax3.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()

    print_insights(df)


def print_insights(df):
    """
    Print key insights from the data
    """
    print("\nKEY INSIGHTS:")
    print("="*60)
    if 'DATE' in df.columns:
        print(f"Date Range: {df['DATE'].min()} to {df['DATE'].max()}")
    if 'CLOSE PRICE' in df.columns:
        prices = df['CLOSE PRICE'].dropna()
        if len(prices) > 0:
            print(f"Price Range: ${prices.min():.2f} - ${prices.max():.2f}")
            print(f"Current Price: ${prices.iloc[-1]:.2f}")
    if 'DAILY RETURN %' in df.columns:
        returns = df['DAILY RETURN %'].dropna()
        if len(returns) > 0:
            print(f"Average Daily Return: {returns.mean():.2f}%")
            print(f"Return Volatility: {returns.std():.2f}%")
            if 'DATE' in df.columns:
                max_idx = returns.idxmax(); min_idx = returns.idxmin()
                if pd.notna(max_idx) and pd.notna(min_idx):
                    print(f"Best Day: {returns.max():.2f}% on {df.loc[max_idx, 'DATE']}")
                    print(f"Worst Day: {returns.min():.2f}% on {df.loc[min_idx, 'DATE']}")

## Step 2: Load the sheet and run the full analysis

This reads your data (or sample data), cleans the columns, and produces an interactive Plotly dashboard, static charts, and a printed insights summary.

In [ ]:
# ============================================
# MAIN EXECUTION
# ============================================

# Paste your own Google Sheet URL (the sheet your n8n pipeline writes to)
your_sheet_url = 'https://docs.google.com/spreadsheets/d/PASTE_YOUR_SHEET_ID_HERE/edit'

df = load_google_sheet_data(sheet_url=your_sheet_url)

if df is not None:
    df = analyze_stock_data(df)
else:
    print("Could not load data for analysis")

---
### What you learned

You connected an **automated pipeline** (n8n + Polygon.io + Google Sheets) to a Colab analysis: the pipeline *collects* data continuously, and your notebook *analyzes* whatever it has gathered.

— eMathrix Education · Data Science & Data Analytics